# Gaussian Continuation Data Preparation
This notebook prepares datasets for Gaussian continuation experiments using the Ackley function. Steps:
1. Generate $(X, Y_{raw})$ using the Ackley function.
2. Compute the pairwise distance matrix for $X$.
3. Select a sequence of smoothing sigmas.
4. For each sigma, compute the Gaussian kernel, normalize, and generate smoothed targets.
5. Store each $(X, Y_{\sigma_k})$ for experimentation.

In [ ]:
import numpy as np
from ma_thesis.data import ackley, levy, eggholder

def generate_and_smooth(
    func, x_range, y_range, N=2000, K=6, sigma_scale=5, seed=42
):
    np.random.seed(seed)
    X = np.random.uniform(x_range[0], x_range[1], size=(N, 2))
    Y_raw = func(X)
    from sklearn.neighbors import NearestNeighbors
    neigh = NearestNeighbors(n_neighbors=N, algorithm='auto')
    neigh.fit(X)
    D_dist = neigh.kneighbors_graph(X, mode='distance').toarray()
    nearest_distances, _ = neigh.kneighbors(X, n_neighbors=2)
    mean_nn_dist = np.mean(nearest_distances[:, 1])
    sigma_max = sigma_scale * mean_nn_dist
    sigmas = np.linspace(sigma_max, 0, K)
    datasets = []
    for sigma in sigmas:
        if sigma > 0:
            W = np.exp(-D_dist**2 / (2 * sigma**2))
        else:
            W = np.eye(N)
        W_norm = W / W.sum(axis=1, keepdims=True)
        Y_sigma = W_norm @ Y_raw
        datasets.append((X.copy(), Y_sigma.copy()))
    return X, Y_raw, sigmas, datasets


In [ ]:
# Example usage for Ackley, Levy, Eggholder
functions = [
    (ackley, (-5, 5), (-5, 5), "Ackley"),
    (levy, (-10, 10), (-10, 10), "Levy"),
    (eggholder, (-512, 512), (-512, 512), "Eggholder")
]

results = {}
for func, x_rng, y_rng, name in functions:
    X, Y_raw, sigmas, datasets = generate_and_smooth(func, x_rng, y_rng)
    results[name] = dict(X=X, Y_raw=Y_raw, sigmas=sigmas, datasets=datasets, x_range=x_rng, y_range=y_rng)
    print(f"{name}: X shape {X.shape}, Y_raw shape {Y_raw.shape}, sigmas: {sigmas}")

In [ ]:
from scipy.interpolate import griddata
import matplotlib.pyplot as plt

def plot_all_sigmas_surface_and_scatter(results, grid_res=80):
    for name, res in results.items():
        X, sigmas, datasets, x_range, y_range = res['X'], res['sigmas'], res['datasets'], res['x_range'], res['y_range']
        n = len(sigmas)
        fig = plt.figure(figsize=(12, 4 * n))
        fig.suptitle(name, fontsize=16)
        for i in range(n):
            X_plot, Y_plot = datasets[i]
            # Surface plot
            ax1 = fig.add_subplot(n, 2, 2*i+1, projection='3d')
            xg = np.linspace(x_range[0], x_range[1], grid_res)
            yg = np.linspace(y_range[0], y_range[1], grid_res)
            Xg, Yg = np.meshgrid(xg, yg)
            Zg = griddata(X_plot, Y_plot, (Xg, Yg), method='cubic', fill_value=np.nan)
            surf = ax1.plot_surface(Xg, Yg, Zg, cmap='viridis', edgecolor='none', alpha=0.95)
            ax1.set_title(f"Sigma = {sigmas[i]:.3f}\nSurface plot")
            ax1.set_xlabel("x1")
            ax1.set_ylabel("x2")
            ax1.set_zlabel("f(x)")
            # Scatter plot
            ax2 = fig.add_subplot(n, 2, 2*i+2, projection='3d')
            ax2.scatter(X_plot[:, 0], X_plot[:, 1], Y_plot, c=Y_plot, cmap='viridis', s=10)
            ax2.set_title(f"Sigma = {sigmas[i]:.3f}\nScatter plot")
            ax2.set_xlabel("x1")
            ax2.set_ylabel("x2")
            ax2.set_zlabel("f(x)")
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

plot_all_sigmas_surface_and_scatter(results)